# Delta-V budget

A worked example of the `quicksat` delta-V budget against the sample satellite: every manoeuvre the mission has to fly, what each costs, and how much propellant the total implies.

For *why* each step works the way it does — the closed forms, why collision avoidance has no physics of its own, what the circular assumption costs — see `docs/delta_v_ref.ipynb`. For the mission the manoeuvres are flown from, see `docs/mission_ref.ipynb`.

In [1]:
# Change log. LAST_CHANGE and CHANGE_NOTE are typed in by hand: update them whenever
# the inputs move -- a manoeuvre added, a contact count renegotiated, a mission life
# extended -- so that the figures below can be read against what produced them. The
# run time is recorded automatically, and says how stale the outputs stored in this
# notebook are relative to that last change.
from datetime import datetime

LAST_CHANGE = "2026-09-21"
CHANGE_NOTE = "Collision avoidance is a one-way hop; disposal a 500 x 250 km orbit."

print(f"last change  {LAST_CHANGE}")
print(f"             {CHANGE_NOTE}")
print(f"last run     {datetime.now().astimezone():%Y-%m-%d %H:%M %Z}")

last change  2026-09-21
             Collision avoidance is a one-way hop; disposal a 500 x 250 km orbit.
last run     2026-09-23 23:33 CEST


In [2]:
import os
from pathlib import Path

import pandas as pd

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from sample/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat.delta_v.budget import DeltaVBudget
from quicksat.mass.budget import MassBudget
from quicksat.utils.mission import Mission

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## The inputs

A manoeuvre list, a config, and the shared mission. The manoeuvres are flown from its orbit and the recurring ones are scaled by its duration, so it is loaded once and handed over rather than re-read. The mass budget goes in the same way: the spacecraft is an input to the sizing, and attaching it lets `propellant_mass()` take the dry mass from there.

In [3]:
DATA = Path("sample") / "data"
mission = Mission.from_yaml_file(DATA / "mission.yaml")
mass_data = MassBudget.from_csv(DATA / "equipment.csv", DATA / "mass_budget_config.yaml")
dv_data = DeltaVBudget.from_csv(
    DATA / "manoeuvres.csv",
    DATA / "delta_v_config.yaml",
    DATA / "mission.yaml",
    mass_budget=mass_data,
)

config = dv_data.config
print(f"mission    {mission.altitude:~.0f} circular, velocity {mission.velocity:~.3f}")
print(f"           design life {mission.duration:~}")
print(f"propulsion Isp {config.propulsion.isp:~}")
print(f"margin     {config.margin:g}%")
print(f"CAM        {'hop up and back down' if config.collision_avoidance.return_burn else 'one-way hop'}")
print()
dv_data.manoeuvre_table[
    ["manoeuvre_id", "phase", "manoeuvre_type", "count", "recurring", "loss_factor"]
]

mission    500 km circular, velocity 7.613 km / s
           design life 7 a
propulsion Isp 220 s
margin     5%
CAM        one-way hop



,manoeuvre_id,phase,manoeuvre_type,count,recurring,loss_factor
0,injection_correction,Commissioning,altitude_change,1.00,False,1.00
1,inclination_trim,Commissioning,inclination_change,1.00,False,1.00
2,phasing,Commissioning,given,1.00,False,1.00
3,drag_makeup,Operations,altitude_change,1.00,True,1.00
4,collision_avoidance,Operations,collision_avoidance,4.00,True,1.00
5,inclination_maint,Operations,inclination_change,1.00,True,1.00
6,deorbit,Disposal,deorbit,1.00,False,1.00


## What each manoeuvre costs

`resolve()` turns each row into a delta-V, scaled by that row's `loss_factor` — 1.0 throughout here, since no finite-burn analysis has been done yet. `deltav_each` is one manoeuvre; `occurrences` is how many times it happens over the mission, which for a recurring row is the yearly count multiplied by the mission duration.

In [4]:
resolved = dv_data.resolve()
resolved[["manoeuvre_id", "manoeuvre_type", "deltav_each", "occurrences", "deltav_total"]]

,manoeuvre_id,manoeuvre_type,deltav_each,occurrences,deltav_total
0,injection_correction,altitude_change,6.63,1.00,6.63
1,inclination_trim,inclination_change,6.64,1.00,6.64
2,phasing,given,8.00,1.00,8.00
3,drag_makeup,altitude_change,0.66,7.00,4.65
4,collision_avoidance,collision_avoidance,0.11,28.00,3.10
5,inclination_maint,inclination_change,1.59,7.00,11.16
6,deorbit,deorbit,70.78,1.00,70.78


## The totals

Disposal is the largest phase, but not by the margin a controlled re-entry would give it. The single burn onto a 500 × 250 km decay orbit costs 71 m/s, against the 40 m/s that commissioning and operations need between them; a direct re-entry from the same orbit would cost 145 and swamp both.

In [5]:
print(f"before margin    {dv_data.total_deltav(margin=False):~.2f}")
print(f"margin ({config.margin:g}%)       "
      f"{(dv_data.total_deltav() - dv_data.total_deltav(margin=False)):~.2f}")
print(f"total            {dv_data.total_deltav():~.2f}")
pd.concat({"by phase": dv_data.by_phase()}, axis=1)

before margin    110.97 m / s
margin (5%)       5.55 m / s
total            116.51 m / s


,by phase
,deltav
phase,
Commissioning,22.34
Disposal,74.32
Operations,19.85


In [6]:
pd.concat({"by type": dv_data.by_type()}, axis=1)

,by type
,deltav
manoeuvre_type,
altitude_change,11.84
collision_avoidance,3.25
deorbit,74.32
given,8.40
inclination_change,18.69


## Propellant, and the sizing loop

The rocket equation turns the budget into a propellant mass from the dry mass and the Isp. With a mass budget attached, `propellant_mass()` takes the dry mass from it; passing one explicitly overrides that, which is how a what-if is asked without editing the equipment file. Nothing is written back either way — the equipment CSV stays the source of truth for what is actually loaded, and the comparison below is left to a human.

In [7]:
dry_mass = mass_data.in_orbit_mass(propellant=0)
required = dv_data.propellant_mass()

loaded = mass_data.propellant_mass()

print(f"in-orbit dry mass    {dry_mass:~.2f}")
print(f"delta-V to deliver   {dv_data.total_deltav():~.2f}")
print(f"propellant required  {required:~.2f}")
print(f"propellant loaded    {loaded:~.2f}   (the equipment CSV)")
print()
shortfall = required - loaded
if shortfall > 0:
    print(f"SHORT by {shortfall:~.2f} - the equipment list does not carry enough")
else:
    print(f"enough, with {abs(shortfall):~.2f} to spare")

in-orbit dry mass    448.62 kg
delta-V to deliver   116.51 m / s
propellant required  24.89 kg
propellant loaded    22.00 kg   (the equipment CSV)

SHORT by 2.89 kg - the equipment list does not carry enough


## The budget, in one table

`tabulated_deltav()` renders the whole budget as a document: every manoeuvre grouped into its mission phase with a subtotal, then the total, the margin, and the total including it.

In [8]:
dv_data.tabulated_deltav()

Item,Name,Type,Input,ΔV each [m/s],Times,ΔV total [m/s]
injection_correction,Launcher dispersion correction,altitude_change,12 km,6.632,1.0,6.63
inclination_trim,Injection inclination trim,inclination_change,0.05 deg,6.643,1.0,6.64
phasing,Orbit phasing to the reference slot,given,8 m / s,8.000,1.0,8.00
,Commissioning subtotal,,,,,21.28
drag_makeup,Drag make-up,altitude_change,1.2 km,0.664,7.0,4.65
collision_avoidance,Collision avoidance,collision_avoidance,200 m,0.111,28.0,3.10
inclination_maint,Inclination maintenance,inclination_change,0.012 deg,1.594,7.0,11.16
,Operations subtotal,,,,,18.91
deorbit,End-of-life deorbit,deorbit,250 km,70.783,1.0,70.78
,Disposal subtotal,,,,,70.78
